# Reproducing Ishii et al. (2025) from dandiset 001249 and the authors' Figshare deposit

Paper: Ishii, Cortese, Leng, Lee, Bayless, Bhattacharya, Muir, Stuber (2025), "Post-ejaculatory
inhibition of female sexual drive via heterogeneous neuronal ensembles in the medial preoptic
area", eLife 12, RP91765, doi:10.7554/elife.91765.

Dataset: DANDI:001249, version 0.241109.2357 (doi:10.48324/dandi.001249/0.241109.2357), 14 NWB
files of whole-brain light-sheet volumes, one per brain. The paper does not cite a version of the
dandiset; this is the only published version (the draft has the same 14 assets and was last
modified before it was published), so it is the version pinned (protocol, author step 1.3).

Outside data: The claims file marks every claim outside Figure 2 `data: elsewhere`: the paper
deposited those data on Figshare, doi:10.6084/m9.figshare.27642768.v1 (version 1), read here in
place. The MD5 of each zip read is printed below and must match `mpoa.FIGSHARE_FILES`.

Text: The claims file was written from the version of record (eLife RP91765, 5 March 2025),
with the code at github.com/stuberlab/Ishii-MPOA-Ejaculation-2023, commit 4e98c43. The code's
notebooks are named by the preprint's figures: "Figure S5" is the published Figure 3, "Figure 3,
4, 5, S6, S7, S8" the published Figures 4 to 6, "Figure 6" the published Figure 7, and "Figure
S9", "S10", "S11" the published Figure 7, figure supplements 1 to 3. The Figshare zips use the
same names.

| Claim | Data | Tested here |
|---|---|---|
| fig2-density-link, fig2c-brain-wide-increase, fig2ei-social-regions-higher, fig2-mpn-higher, fig2f-mpn-fourth-largest, fig2gh-consummatory-tendency, fig2s-dorsal-raphe | dandiset | blocked, too large (cost measured below) |
| fig1gh-transitions-isolation, fig1i-latency-return, fig1jm-behaviors-after-ejaculation, fig1nt-natural-estrus, fig1s1-experience, fig1s2-novel-male | Figshare | tested |
| fig3d-mpn-fos, fig3i-vgat-vglut2-fos, fig3jm-cell-types | Figshare | tested |
| fig4h-response-magnitude, fig4i-ejaculation-responders, fig5-ejaculation-specific, fig5c-class-selectivity, fig6c-decay-length, fig6k-late-cluster-vgat | Figshare | tested |
| fig7fg-activation-suppresses, fig7kl-activation-intromission, fig7s1-mpn-proportion, fig7s2-home-cage-activation, fig7s3-inhibition | Figshare | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_chisquare

import mpoa

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print("dandiset", mpoa.DANDISET, "version", mpoa.VERSION, "| Figshare", mpoa.FIGSHARE_DOI)

## Data check

### The dandiset

Every file is opened by streaming. For each, the metadata, the shape of both image series, and
one plane from the middle of the stack (TRAP channel in full, background channel as a central
1024 x 1024 crop) are read: their intensity ranges and a checksum of the TRAP plane, to look for
brains deposited twice. The time to read a plane gives the cost of the whole-brain analysis.

In [ ]:
assets = mpoa.dandi_assets()
assets["group"] = assets.path.str.extract(r"sub-(\w+)-\d")[0]
print(len(assets), "assets,", round(assets["size"].sum() / 1e12, 3), "TB")
print(assets.groupby("group").size().rename("brains").to_string())
import hashlib
rows, read_s, read_mb = [], 0.0, 0.0
for a in assets.itertuples():
    f = mpoa.open_nwb(a.s3_url)
    trap, bg = f["acquisition/TRAP"], f["acquisition/background"]
    zmid = trap["data"].shape[0] // 2
    t = time.time()
    plane = trap["data"][zmid]
    read_s += time.time() - t
    read_mb += plane.nbytes / 1e6
    y0, x0 = bg["data"].shape[1] // 2 - 512, bg["data"].shape[2] // 2 - 512
    crop = bg["data"][zmid, y0:y0 + 1024, x0:x0 + 1024]
    rows.append(dict(
        path=a.path.split("/")[1][:40], subject=f["general/subject/subject_id"][()].decode(),
        session_start=f["session_start_time"][()].decode()[:10],
        description=f["session_description"][()].decode(),
        shape=trap["data"].shape, dtype=str(trap["data"].dtype), bg_shape=bg["data"].shape,
        chunks=trap["data"].chunks, compression=trap["data"].compression,
        rate=trap["starting_time"].attrs["rate"], resolution=trap["data"].attrs["resolution"],
        unit=trap["data"].attrs["unit"], trap_mid_mean=plane.mean(), trap_mid_p999=np.percentile(plane, 99.9),
        bg_crop_mean=crop.mean(), trap_md5=hashlib.md5(plane.tobytes()).hexdigest()[:10],
        groups=sorted(k for k in f.keys()), processing=list(f["processing"].keys()),
        analysis=list(f["analysis"].keys())))
    f.close()
nwb = pd.DataFrame(rows)
print(nwb[["path", "subject", "session_start", "shape", "bg_shape", "dtype", "chunks", "compression",
           "rate", "resolution", "unit"]].to_string())
print(nwb[["subject", "description", "trap_mid_mean", "trap_mid_p999", "bg_crop_mean", "trap_md5",
           "processing", "analysis"]].to_string())
print("distinct TRAP plane checksums:", nwb.trap_md5.nunique(), "of", len(nwb))
print("top-level groups:", nwb.groups.iloc[0])
rate = read_mb / read_s
vox = sum(np.prod(s) for s in nwb["shape"]) * 2 / 1e9
print(f"streaming rate {rate:.1f} MB/s over {read_mb:.0f} MB")
print(f"TRAP channels {vox:.0f} GB; both channels {2 * vox:.0f} GB")
print(f"time to stream the TRAP channels once: {vox * 1e3 / rate / 3600:.2f} h; both channels "
      f"{2 * vox * 1e3 / rate / 3600:.2f} h")

### The Figshare deposit

The version's file list, with the MD5 of each zip against the one pinned in `mpoa.py`, and the
tables read, with their counts.

In [ ]:
fl = mpoa.figshare_file_list()
fl["pinned_md5"] = fl.name.str.replace(".zip", "", regex=False).map(
    {k: v[1] for k, v in mpoa.FIGSHARE_FILES.items()})
fl["md5_matches"] = fl.md5 == fl.pinned_md5
print(fl[["id", "name", "size", "md5", "md5_matches"]].to_string())
assert fl.md5_matches.all()

F1 = "Figure 1 and S1,S2"
g1 = mpoa.read_csv(F1, "data/group_df.csv")
m1 = mpoa.read_csv(F1, "data/merge_df.csv")
lat1 = mpoa.read_csv(F1, "data/latency_df.csv")
es = mpoa.read_csv(F1, "data/estrus_groupdf.csv")
es_meta = mpoa.read_csv(F1, "meta/Figure1_Estrus_meta.csv")
vse = mpoa.read_csv(F1, "data/group_df_virgin_SE.csv")
vse_m = mpoa.read_csv(F1, "data/merge_df_virgin_SE.csv")
rex = mpoa.read_csv(F1, "data/reexposure_df.csv")
print("Figure 1, hormone primed: animals", g1.ID.nunique(), "| rows by State",
      g1.State.value_counts().to_dict())
print("  zone visits (merge_df) by Type", m1.Type.value_counts().to_dict(), "| animals", m1.ID.nunique())
print("  latency_df animals", len(lat1), "| range (s)", lat1.iloc[:, 1:].min().round(1).to_dict(),
      lat1.iloc[:, 1:].max().round(1).to_dict())
print("Figure 1, natural estrus: animals", es.ID.nunique(), "| rows by state",
      es.sort_state_name.value_counts().to_dict(), "| latency meta animals", es_meta.ID.nunique())
print("Figure 1-S1: animals by group and state")
print(vse.groupby(["Estrus_state", "State"]).ID.nunique().unstack().to_string())
print("  visit table animals", vse_m.ID.nunique(), "| columns", [c for c in vse_m.columns][:12])
print("Figure 1-S2: animals", rex.ID.nunique(), "| rows by assay", rex.Rexposure.value_counts().to_dict())

# Check of the per-10-min isolation zone entries in group_df against the visits in merge_df,
# and of the per-visit mean duration against Time_in_Isolation_zone_per_10min.
iso = m1[m1.Type == "Isolation_zone"].groupby(["ID", "State"]).agg(
    n_visits=("Duration_second", "size"), mean_visit_s=("Duration_second", "mean"),
    total_s=("Duration_second", "sum")).reset_index()
chk = g1.merge(iso, on=["ID", "State"], how="left")
print("isolation visits in merge_df against Number_of_Isolation_zone_entry (group_df): equal in",
      int((chk.n_visits == chk.Number_of_Isolation_zone_entry).sum()), "of", len(chk), "rows")
print("Duration (s) of each epoch by State:", g1.groupby("State").Duration.agg(["min", "max"]).round(0).to_dict())
print("Time_in_Isolation_zone_per_10min against total visit time x 600 / epoch duration: max abs diff",
      round(float(np.nanmax(abs(chk.total_s * 600 / chk.Duration - chk.Time_in_Isolation_zone_per_10min))), 3))
print("Time_in_Isolation_zone_per_10min against the mean isolation visit duration: max abs diff",
      float(np.nanmax(abs(chk.mean_visit_s - chk.Time_in_Isolation_zone_per_10min))))
print("so the column named per 10 min holds the mean visit duration in s")

## Figure 1: sexual motivation after male ejaculation (Figshare)

Tests as Figure 1.ipynb runs them: `AnovaRM` with one row per animal and epoch, then
`pairwise_tukeyhsd` over the epochs (which treats them as independent groups), and scipy's
`wilcoxon` for pre against post. Pairs are matched by animal id here; the code pairs by row
order, and the check below shows the two agree. scipy 1.18 picks the exact or the normal
approximation by itself; where the data have ties or zero differences the printed p of the
paper came from the normal approximation, so both are printed.

In [ ]:
def rm_and_tukey(df, value, subject, within, label):
    r = mpoa.rm_anova(df, value, subject, within)
    print(f"{label}: RM ANOVA F({r['df1']:.0f},{r['df2']:.0f}) = {r['F']:.3f}, p = {r['p']:.3g}")
    means = df.groupby(within)[value].mean()
    print("  means", means.round(3).to_dict())
    print(mpoa.tukey(df[value], df[within]).to_string(index=False))
    return r


def wilcox(x, y):
    out = {}
    for m in ["auto", "approx"]:
        out[m] = stats.wilcoxon(x, y, method=m).pvalue
    return out

### fig1gh-transitions-isolation

> In hormone-primed females, zone transitions decrease and time in the isolation zone increases
> after male ejaculation, compared with before ejaculation and with the control trial.

Transitions: the code's column, isolation zone entries per 10 min; the legend's "number of
transitions between the zones" read as entries into either zone (`Number_of_Entry_per_10min`)
is shown as a variant. Time: the legend's "average time spent in the isolation zone", which the
code computes as the mean visit duration from merge_df. The column named time per 10 min holds
the same values (data check), so it is not a separate reading.

In [ ]:
iso_long = iso.rename(columns={"mean_visit_s": "ave_iso_s"})
print("animals with an isolation visit in every epoch:",
      int((iso_long.groupby("ID").size() == 3).sum()), "of", g1.ID.nunique())
r_tr = rm_and_tukey(g1, "Number_of_Isolation_zone_entry_per_10min", "ID", "State",
                    "transitions (isolation zone entries per 10 min)")
both = m1.groupby(["ID", "State"]).size().rename("n_visits_both").reset_index().merge(
    g1[["ID", "State", "Duration"]], on=["ID", "State"])
both["visits_both_per_10min"] = both.n_visits_both * 600 / both.Duration
print("Number_of_Entry_per_10min is empty for", int(g1.Number_of_Entry_per_10min.isna().sum()),
      "rows; the variant counts visits to either zone in merge_df, per 10 min of the epoch")
r_tr2 = rm_and_tukey(both, "visits_both_per_10min", "ID", "State",
                     "variant, visits to either zone per 10 min")
r_iso = rm_and_tukey(iso_long, "ave_iso_s", "ID", "State", "average isolation zone visit (s)")
iso_long["total_per_10min"] = iso_long.total_s * 600 / iso_long.merge(
    g1[["ID", "State", "Duration"]], on=["ID", "State"]).Duration.values
r_iso2 = rm_and_tukey(iso_long, "total_per_10min", "ID", "State",
                      "variant, total time in isolation zone per 10 min of the epoch (s)")
print("per animal:")
print(g1.pivot(index="ID", columns="State", values="Number_of_Isolation_zone_entry_per_10min").round(2)
      .join(iso_long.pivot(index="ID", columns="State", values="ave_iso_s").add_suffix("_iso_s").round(1)).to_string())

**Result: reproduced.** Isolation zone entries per 10 min fall from 5.33 (pre) and 6.40 (control) to 1.79 after ejaculation, RM ANOVA F(2,16) = 10.888, p = 0.00104, Tukey post against pre p = 0.0142 and control against post p = 0.0015. The average isolation visit rises from 128.6 s (pre) and 47.1 s (control) to 389.1 s, F(2,16) = 14.344, p = 0.00027, Tukey 0.0057 and 0.0004. These are the legend's values. Counting visits to either zone gives the same pattern (F = 6.714, p = 0.0076; post against pre 0.0210, control against post 0.0114). The total time in the isolation zone per 10 min of the epoch, which is not the legend's quantity, is higher before ejaculation than in the control trial and does not differ between pre and post (Tukey p = 0.777).

### fig1i-latency-return

> Latency to return to the interaction zone is longest after male ejaculation, longer than after
> sniffing or mounting.

latency_df.csv, one value per animal for sniff (`Interaction_latency`), mount and ejaculation.
The latencies are capped at 600 s (the post-ejaculation epoch); the number at the cap is printed.

In [ ]:
lat_long = lat1.melt(id_vars="filename", var_name="Behavior", value_name="latency_s")
print("values at 600 s by behavior:", lat_long[lat_long.latency_s >= 600].Behavior.value_counts().to_dict())
r_lat = rm_and_tukey(lat_long, "latency_s", "filename", "Behavior", "latency to return (s)")
print(lat1.round(1).to_string(index=False))

**Result: reproduced.** Mean latency to return is 438.3 s after ejaculation, 105.2 s after sniff and 159.5 s after mount; RM ANOVA F(2,16) = 11.870, p = 0.000691; Tukey ejaculation against sniff p = 0.0016 and against mount p = 0.0077, as in the legend. Four of the nine ejaculation latencies sit at the 600 s cap.

### fig1jm-behaviors-after-ejaculation

> After ejaculation, female-to-male sniffing does not change, while mounting and intromission
> decrease and self-grooming increases (hormone-primed females).

In [ ]:
rows = []
for name, col in [("sniff", "Number_of_Sniff_START_per_10min"), ("mount", "Number_of_Mounting_START_per_10min"),
                  ("intromission", "Number_of_Intromission_START_per_10min"),
                  ("grooming", "Number_of_Grooming_START_per_10min")]:
    pre, post, ids = mpoa.paired(g1, "ID", "State", "Pre_Ejaculation", "Post_Ejaculation", col)
    rowpre = g1[g1.State == "Pre_Ejaculation"][col].values
    rowpost = g1[g1.State == "Post_Ejaculation"][col].values
    p = wilcox(pre, post)
    rows.append(dict(behavior=name, n=len(ids), pre_mean=pre.mean(), post_mean=post.mean(),
                     n_lower_after=int((post < pre).sum()), n_higher_after=int((post > pre).sum()),
                     p_auto=p["auto"], p_approx=p["approx"],
                     row_order_p=stats.wilcoxon(rowpre, rowpost, method="approx").pvalue))
jm = pd.DataFrame(rows)
print(jm.round(4).to_string(index=False))

**Result: reproduced.** Over 9 mice, mounting and intromission fall to 0 in all 9 (Wilcoxon p = 0.0039, exact, the legend's 0.00390625), self-grooming rises in 7 (3.50 to 13.54 per 10 min, p = 0.0234 exact and 0.0251 by the normal approximation the legend's 0.025061844 comes from), and sniffing does not change (p = 0.652 exact, 0.594 approximate). Pairing by row order, as the code does, gives the same p-values.

### fig1nt-natural-estrus

> Naturally cycling estrus females show the same post-ejaculatory changes, with a strong
> reduction of sniffing and no change in self-grooming.

estrus_groupdf.csv (transitions and average isolation time per animal and epoch), and the
latencies from meta/Figure1_Estrus_meta.csv, as Figure 1.ipynb reads them.

In [ ]:
r_ntr = rm_and_tukey(es, "transition_nper10min", "ID", "sort_state_name", "estrus, transitions per 10 min")
r_niso = rm_and_tukey(es, "ave_time_in_isolation_zone", "ID", "sort_state_name", "estrus, average isolation time (s)")
es_lat = es_meta[["ID", "interaction_ave_latency", "mount_ave_latency", "ejac_ave_latency"]].melt(
    id_vars="ID", var_name="Behavior", value_name="latency_s")
r_nlat = rm_and_tukey(es_lat, "latency_s", "ID", "Behavior", "estrus, latency to return (s)")
rows = []
for name, col in [("sniff", "Sniffing_nper10min"), ("mount", "Mounting_nper10min"),
                  ("intromission", "Intromission_nper10min"), ("grooming", "Grooming_nper10min")]:
    pre, post, ids = mpoa.paired(es, "ID", "sort_state_name", "Pre_Ejaculation", "Post_Ejaculation", col)
    p = wilcox(pre, post)
    rows.append(dict(behavior=name, n=len(ids), pre_mean=pre.mean(), post_mean=post.mean(),
                     n_lower_after=int((post < pre).sum()), p_auto=p["auto"], p_approx=p["approx"]))
nt = pd.DataFrame(rows)
print(nt.round(4).to_string(index=False))

**Result: reproduced.** In 6 naturally estrus females, transitions F(2,10) = 7.542, p = 0.0101 (Tukey post against pre p = 0.0038), average isolation time F(2,10) = 15.714, p = 0.00082, and latency F(2,10) = 38.364, p = 2.04e-05, as in the legend. Sniffing, mounting and intromission are lower after ejaculation in all 6 mice (p = 0.03125 exact, 0.0277 approximate); grooming does not differ (p = 0.4375 exact, 0.3452 approximate, the legend's value).

### fig1s1-experience

> Virgin females spend more time in the isolation zone than sexually experienced females; the
> number of isolation zone entries also differs between the two groups.

Pre-ejaculation epoch of the sexual interaction trial (claims file, from Figure S1.ipynb).
Two-sided Mann-Whitney U. The legend's panel C is "the average time spent in the isolation
zone", computed here as the mean visit duration from merge_df_virgin_SE.csv, as Figure 1.ipynb
computes the same quantity for Figure 1H; Figure S1.ipynb tests the time per 10 min instead,
shown as the second reading. In the hormone-primed table that column holds the mean visit
duration (data check), so the two readings should nearly agree.

In [ ]:
pre_v = vse[vse.State == "Pre_Ejaculation"]
print("visit table states:", vse_m.State.value_counts().to_dict(), "(spelled Pre_ejaculation there)")
vis = vse_m[(vse_m.Type == "Isolation_zone") & (vse_m.State.str.lower() == "pre_ejaculation")].groupby("ID").Duration_second.mean()
pre_v = pre_v.assign(ave_iso_s=pre_v.ID.map(vis))
rows = []
for name, col in [("isolation zone entries per 10 min", "Number_of_Isolation_zone_entry_per_10min"),
                  ("average isolation visit (s), legend", "ave_iso_s"),
                  ("time in isolation zone per 10 min (s), code", "Time_in_Isolation_zone_per_10min")]:
    v = pre_v[pre_v.Estrus_state == "Virgin"][col].dropna()
    s = pre_v[pre_v.Estrus_state == "SE"][col].dropna()
    rows.append(dict(measure=name, n_virgin=len(v), n_SE=len(s), virgin_mean=v.mean(), SE_mean=s.mean(),
                     p=stats.mannwhitneyu(v, s).pvalue))
s1 = pd.DataFrame(rows)
print(s1.round(4).to_string(index=False))

**Result: reproduced.** In the pre-ejaculation epoch, virgin females make fewer isolation zone entries than sexually experienced ones (2.42 against 6.36 per 10 min, Mann-Whitney p = 0.0411, the legend's value). Virgin females stay longer per isolation visit: 545.8 against 72.0 s computed from the visits (p = 0.0411), and 582.4 against 72.0 s in the column Figure S1.ipynb tests (p = 0.0260, the legend's value).

### fig1s2-novel-male

> The post-ejaculatory reduction in sexual motivation does not depend on the state of the male;
> with a novel sexually active male after ejaculation, females still make fewer isolation zone
> entries and spend more time there.

Two-sided Wilcoxon signed-rank, pre- against post-ejaculation assay, paired by animal. The legend
(panel B) is "the number of entries in the isolation zone per 10 min", the column
`Isolation_zone_entry`; Figure S2.ipynb tests `Interaction_zone_entry`, shown as a variant. Time
is `Isolation_zone_avetime` (legend C, "average time spent in the isolation zone").

In [ ]:
rows = []
for name, col in [("isolation zone entries (legend)", "Isolation_zone_entry"),
                  ("interaction zone entries (code)", "Interaction_zone_entry"),
                  ("average isolation visit (s)", "Isolation_zone_avetime")]:
    pre, post, ids = mpoa.paired(rex, "ID", "Rexposure", "Pre_Ejaculation", "Post_Ejaculation", col)
    p = wilcox(pre, post)
    rows.append(dict(measure=name, n=len(ids), pre_mean=pre.mean(), post_mean=post.mean(),
                     n_higher_after=int((post > pre).sum()), n_lower_after=int((post < pre).sum()),
                     p_auto=p["auto"], p_approx=p["approx"]))
s2 = pd.DataFrame(rows)
print(s2.round(4).to_string(index=False))
print(rex[["ID", "Rexposure", "Isolation_zone_entry", "Interaction_zone_entry", "Isolation_zone_avetime"]]
      .sort_values(["ID", "Rexposure"]).round(2).to_string(index=False))

**Result: reproduced.** With a novel male after ejaculation, isolation zone entries fall from 2.33 to 0.43 per 10 min (lower in 6 of 7, none higher; Wilcoxon p = 0.0312 exact, 0.0273 approximate, the legend's 0.027281171), and the average isolation visit rises from 595.7 to 1627.8 s (higher in 6 of 7; p = 0.0312 exact, 0.0277 approximate). The code's column, interaction zone entries, gives the same p-values.

## Figure 3: HCR in situ of Fos and marker genes in the preoptic area (Figshare)

The per-cell table (rawdata/MPOA_female_summary_result.csv, one row per cell, with a 0/1
"positive_cells" call per gene) is read, and per mouse, region and section the proportions
are computed from it, then averaged over sections, as Figure S5.ipynb does from its
precomputed per-section tables (`groupby(['ID', 'Analysis major region', 'Stim']).mean()`).
The data check compares these with the deposited per-section tables. Student's t-test (scipy
`ttest_ind`) with Bonferroni correction over the comparisons of each panel, as the code runs them.

In [ ]:
F5 = "Figure S5"
genes = ["Fos", "Vgat", "Vglut2", "Esr1", "Gal", "Nts", "Calcr", "Prlr"]
t = time.time()
with mpoa.figshare_zip(F5).open(f"{F5}/rawdata/MPOA_female_summary_result.csv") as fh:
    hcr = pd.read_csv(fh, usecols=["ID", "Analysis major region", "Stim", "section"] +
                      [f"{g} positive_cells" for g in genes])
hcr.columns = ["region" if c == "Analysis major region" else c.replace(" positive_cells", "") for c in hcr.columns]
print(f"per-cell HCR table: {len(hcr)} cells, read in {time.time() - t:.0f} s")
print(hcr.groupby(["Stim"]).ID.unique().to_string())
print("cells by region and group:")
print(hcr.groupby(["region", "Stim"]).size().unstack().to_string())
print("sections per mouse and region:", hcr.groupby(["ID", "region"]).section.nunique().value_counts().to_dict())
print("calls are 0/1:", {g: sorted(hcr[g].unique().tolist()) for g in genes})
meta5 = mpoa.read_csv(F5, "meta/metainfo.csv")
print(meta5[["ID", "Stim", "Original_ID"]].to_string(index=False))
sec_mean = mpoa.read_csv(F5, "rawdata/MPOA_female_summary_result_mean.csv")
vv = mpoa.read_csv(F5, "rawdata/MPOA_female_Vgat_Vglut2_summary_result.csv")
seu = mpoa.read_csv(F5, "rawdata/MPOA_female_summary_regressed_Seurat_MPN.csv")
print("deposited per-section mean table:", sec_mean.shape, "| Vgat/Vglut2 table:", vv.shape,
      "| Seurat MPN cells:", seu.shape, seu.cluster_name.value_counts().to_dict())

In [ ]:
def per_section(df, num, den=None):
    """Per mouse, region and section: share of cells with all of `num` among cells with all of
    `den` (all cells when den is None)."""
    d = df if den is None else df[df[den].all(axis=1)]
    return d.assign(hit=d[num].all(axis=1).astype(float)).groupby(
        ["ID", "region", "Stim", "section"]).hit.mean().rename("value").reset_index()


def per_mouse(sec):
    return sec.groupby(["ID", "region", "Stim"]).value.mean().reset_index()


def ttests(pm, key, label):
    rows = []
    for k, d in pm.groupby(key):
        a = d[d.Stim == "Appetitive"].value
        b = d[d.Stim == "Post_ejaculatory"].value
        r = stats.ttest_ind(a, b)
        rows.append(dict(**{label: k}, n_app=len(a), n_post=len(b), app_mean=a.mean(), post_mean=b.mean(),
                         t=r.statistic, df=len(a) + len(b) - 2, p=r.pvalue))
    out = pd.DataFrame(rows)
    out["p_bonf"] = mpoa.bonferroni(out.p)
    return out

In [ ]:
fos_sec = per_section(hcr, ["Fos"])
cmp = fos_sec.merge(sec_mean.rename(columns={"Analysis major region": "region"})[["ID", "region", "Stim", "Fos positive_cells"]],
                    on=["ID", "region", "Stim"])
print("sections in the per-cell table:", len(fos_sec), "| in the deposited mean table:", len(sec_mean))
print("Fos+ share per section, from cells against the deposited mean table (merged on mouse and region, "
      "the deposited table has no section column):")
print("  per mouse and region mean from cells vs deposited mean of sections, max abs diff",
      float(abs(per_mouse(fos_sec).set_index(["ID", "region"]).value -
                sec_mean.rename(columns={"Analysis major region": "region"}).groupby(["ID", "region"])["Fos positive_cells"].mean()).max()))
mpn = hcr[hcr.region == "MPN"].copy()
# exclusive populations: Vgat+ without Vglut2, and Vglut2+ without Vgat
mpn["VgatX"] = mpn.Vgat & ~mpn.Vglut2
mpn["Vglut2X"] = mpn.Vglut2 & ~mpn.Vgat
print("MPN cells with both Vgat and Vglut2 calls:", int((mpn.Vgat & mpn.Vglut2).sum()), "of", len(mpn))
vv_mpn = vv[vv["Analysis major region"] == "MPN"]
for pop in ["Vgat", "Vglut2"]:
    dep = vv_mpn.groupby("ID")[f"{pop}+ {pop}+ Fos+ positive_cells proportion over gene"].mean()
    for lab, p in [("inclusive", pop), ("exclusive", pop + "X")]:
        mine = per_mouse(per_section(mpn, [p, "Fos"], [p])).set_index("ID").value
        print(f"MPN {pop}+ Fos+ share per mouse, {lab} population from cells vs the deposited table "
              f"(which Figure S5.ipynb reads): max abs diff {float(abs(mine - dep.loc[mine.index]).max()):.3g}")

### fig3d-mpn-fos

> By HCR, the proportion of Fos+ cells is higher after ejaculation than in the appetitive group
> in the MPN but not in nearby regions (MPA, BNST, LPO, VLPO).

In [ ]:
d3 = ttests(per_mouse(fos_sec), "region", "region")
print(d3.round(4).to_string(index=False))
print("per mouse Fos+ share in the MPN:")
print(per_mouse(fos_sec)[lambda x: x.region == "MPN"].round(3).to_string(index=False))

**Result: reproduced.** The Fos+ share in the MPN is 0.170 after ejaculation against 0.090 in the appetitive group, t(8) = -4.2709, Bonferroni p = 0.0136; MPA t(8) = -1.8413, p = 0.514; BNST, LPO and VLPO p = 1. These are the legend's values, recomputed from the per-cell table. All five post-ejaculatory mice are above four of the five appetitive mice in the MPN.

### fig3i-vgat-vglut2-fos

> In the MPN, both Vglut2+ and Vgat+ cells expressing Fos increase after ejaculation, with a
> higher proportion of Fos+ cells in the Vgat population.

Share of Fos+ cells among Vgat+ (or Vglut2+) cells, Bonferroni over the two populations (Figure
S5.ipynb). The paper does not say whether a cell with both Vgat and Vglut2 calls belongs to
either population; the table Figure S5.ipynb reads counts only cells with one of the two (data
check), which is followed (reader step 2: the code settles what the Methods leave open), with
the inclusive populations as a variant. The Vgat against Vglut2 comparison has no test in the paper; it is judged on the means.

In [ ]:
pm_i = pd.concat([per_mouse(per_section(mpn, [g + "X", "Fos"], [g + "X"])).assign(pop=g) for g in ["Vglut2", "Vgat"]])
d3i = ttests(pm_i, "pop", "population")
print("exclusive populations (main):")
print(d3i.round(4).to_string(index=False))
pm_i2 = pd.concat([per_mouse(per_section(mpn, [g, "Fos"], [g])).assign(pop=g) for g in ["Vglut2", "Vgat"]])
d3i2 = ttests(pm_i2, "pop", "population")
print("variant, inclusive populations:")
print(d3i2.round(4).to_string(index=False))
print("post-ejaculatory mean Fos+ share, Vgat", round(d3i.set_index("population").post_mean["Vgat"], 3),
      "| Vglut2", round(d3i.set_index("population").post_mean["Vglut2"], 3))

**Result: reproduced.** With the exclusive populations of the code's table, the Fos+ share rises in Vgat cells from 0.132 to 0.235 (t(8) = -3.5361, Bonferroni p = 0.0153) and in Vglut2 cells from 0.095 to 0.160 (t(8) = -3.3885, p = 0.0190), the legend's values. After ejaculation the share is higher in Vgat (0.235) than in Vglut2 cells (0.160). With inclusive populations both are significant as well (p = 0.0131 and 0.0091), and Vgat stays higher (0.251 against 0.209).

### fig3jm-cell-types

> Fos induction after ejaculation is found in Vglut2 cells only when they co-express Calcr, and
> in Vgat cells co-expressing Calcr, Gal or Prlr; among clusters, the Vglut2, Vgat and Gal
> types have more Fos+ cells after ejaculation, while the Nts type does not.

Marker subsets: share of Fos+ among Vgat+ gene+ (or Vglut2+ gene+) cells in the MPN, Bonferroni
over the five genes within each population (Figure S5.ipynb), with the exclusive populations of
fig3i-vgat-vglut2-fos and the inclusive ones as a variant. Clusters: the `cluster_name` of
the deposited Seurat table (the step that names the clusters is not in the code), Fos+ share per
mouse within each cluster, Bonferroni over the six clusters including N.A., as the code runs it.

In [ ]:
def marker_tests(suffix):
    rows = []
    for pop in ["Vgat", "Vglut2"]:
        pm = pd.concat([per_mouse(per_section(mpn, [pop + suffix, g, "Fos"], [pop + suffix, g])).assign(gene=g)
                        for g in ["Esr1", "Gal", "Nts", "Calcr", "Prlr"]])
        r = ttests(pm, "gene", "gene")
        r.insert(0, "population", pop)
        rows.append(r)
    return pd.concat(rows)
d3j = marker_tests("X")
print("exclusive populations (main):")
print(d3j.round(4).to_string(index=False))
d3j2 = marker_tests("")
print("variant, inclusive populations:")
print(d3j2.round(4).to_string(index=False))
print("cells per subset in the MPN (all mice):",
      {f"{p}+{g}+": int(mpn[[p, g]].all(axis=1).sum()) for p in ["Vgat", "Vglut2"] for g in ["Calcr", "Gal", "Prlr", "Nts", "Esr1"]})
cl = seu.groupby(["ID", "cluster_name", "Stim"]).Fos_positive_cells.mean().rename("value").reset_index()
print("mice per cluster and group:", cl.groupby(["cluster_name", "Stim"]).ID.nunique().unstack().to_dict())
d3m = ttests(cl.rename(columns={"cluster_name": "cluster"}), "cluster", "cluster")
print(d3m.round(4).to_string(index=False))

**Result: reproduced.** In Vgat cells, Fos rises after ejaculation in the Calcr (Bonferroni p = 0.0318), Gal (0.0174) and Prlr (0.0212) subsets; in Vglut2 cells only in the Calcr subset (0.0146; Esr1, Gal, Nts and Prlr 0.104 to 0.822). Among the Seurat clusters the Vglut2+ (0.0189), Vgat+ (0.0022) and Gal+ (0.0265) clusters rise and Nts+ does not (0.377). All match the legend; the inclusive populations give the same pattern (Vglut2 Calcr p = 0.0044, other Vglut2 subsets 0.0706 to 0.413).

## Figures 4 to 6: calcium imaging of MPOA Vgat and Vglut2 neurons (Figshare)

The deposit holds, for 793 cells (rows of male_analysis_magnitude.csv, in the column order of the
arrays), peri-event traces for six behaviors (-5 to +15 s, 10 Hz), the ejaculation trace from
-5 to +120 s, and a table of per-cell magnitudes, responder flags, response classes and decay
lengths. The data check shows how far the arrays support the table.

In [ ]:
F4 = "Figure 3, 4, 5, S6, S7, S8"
ca = mpoa.read_csv(F4, "data/male_analysis_magnitude.csv")
params = mpoa.read_pickle(F4, "meta/parameter_dict.pickle")
print("parameter_dict:", params)
BEH = ["Sniff", "Mount", "Intromission", "Male_anogenital_sniffing", "Grooming", "Ejaculation"]
snip = {b: mpoa.read_npy(F4, f"data/{b}_all_norm_zscore_snip_array.npy") for b in BEH}
longer = mpoa.read_npy(F4, "data/Ejaculation_longer_all_norm_zscore_snip_array.npy")
print("cells:", len(ca), ca.Mouse_line.value_counts().to_dict())
print("cells per animal:", ca.groupby(["Mouse_line", "ID"]).size().to_dict())
print("arrays (time x cells):", {b: snip[b].shape for b in BEH}, "| longer", longer.shape)
print("NaN in arrays:", {b: int(np.isnan(snip[b]).sum()) for b in BEH}, int(np.isnan(longer).sum()))
for b in BEH:
    a = snip[b]
    m_arr = mpoa.magnitude(a)
    c = ca[f"{b}_zscore_magnitude"].values
    print(f"{b:25s} baseline mean range {a[:50].mean(0).min():.1e} to {a[:50].mean(0).max():.1e}, "
          f"baseline SD median {np.median(a[:50].std(0)):.3f}; 0-5 s mean of array vs table magnitude: "
          f"r = {np.corrcoef(m_arr, c)[0, 1]:.3f}; table magnitude range {c.min():.2f} to {c.max():.2f}")
k = snip["Ejaculation"][60] / longer[60]
print("Ejaculation 200-frame array / longer array, per cell: constant over time:",
      bool(np.allclose(snip["Ejaculation"], longer[:200] * k, rtol=1e-6, atol=1e-9)),
      "| the factor equals 1 / SD of the longer array over -5 to 0 s:",
      bool(np.allclose(k, 1 / longer[:50].std(0), rtol=1e-6)))
mag_ej = mpoa.magnitude(longer)
print("Ejaculation magnitude, 0-5 s mean of the longer array vs table: max abs diff",
      float(abs(mag_ej - ca.Ejaculation_zscore_magnitude).max()))
for b in BEH:
    pos = ca[f"{b}_zscore_magnitude"] > 2
    neg = ca[f"{b}_zscore_magnitude"] < -2
    print(f"{b:25s} flags = magnitude > 2: {bool((pos == ca[f'{b}_positive_responding_cells']).all())}, "
          f"< -2: {bool((neg == ca[f'{b}_negative_responding_cells']).all())}")
print("decay length (s) in the table: responders", int(ca.Ejaculation_decay_length_seconds.notna().sum()),
      "| range", ca.Ejaculation_decay_length_seconds.min(), "to", ca.Ejaculation_decay_length_seconds.max())

The ejaculation magnitudes of the table are the 0 to +5 s mean of the -5 to +120 s trace, which
is baseline-subtracted (Methods). The 200-frame arrays named "norm_zscore" are rescaled: for
ejaculation (one event per cell) each cell's trace is divided by its SD over -5 to 0 s, and for
the other behaviors (several events) the baseline SD is below 1, as if each event were scaled
before averaging. Their 0 to 5 s means do not give the table's magnitudes for any behavior.
The traces the Methods' magnitude is computed from (z-scored on the background period,
averaged over events) are deposited only for ejaculation, so for the other five behaviors the
magnitudes are taken from the table, and the rescaled arrays are a variant.

### fig4h-response-magnitude

> Vgat cells show higher average response magnitude than Vglut2 cells after sniffing,
> self-grooming, male anogenital sniffing and ejaculation, but not after mounting or
> intromission.

Student's t-test (scipy `ttest_ind`) per behavior, Bonferroni over the six (legend, Figure 3, 4,
5, S6, S7, S8.ipynb). Ejaculation magnitude from the longer trace; the others from the table.

In [ ]:
mag = pd.DataFrame({b: ca[f"{b}_zscore_magnitude"].values for b in BEH})
mag["Ejaculation"] = mag_ej
vg = (ca.Mouse_line == "Vgat-Cre").values
def line_tests(m):
    rows = []
    for b in BEH:
        r = stats.ttest_ind(m[b][vg], m[b][~vg])
        rows.append(dict(behavior=b, vgat_mean=m[b][vg].mean(), vglut2_mean=m[b][~vg].mean(),
                         t=r.statistic, df=vg.sum() + (~vg).sum() - 2, p=r.pvalue))
    out = pd.DataFrame(rows)
    out["p_bonf"] = mpoa.bonferroni(out.p)
    return out
h = line_tests(mag)
print(h.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("variant, magnitudes from the rescaled 200-frame arrays:")
mag_arr = pd.DataFrame({b: mpoa.magnitude(snip[b]) for b in BEH})
print(line_tests(mag_arr).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("by animal, mean magnitude:")
print(mag.assign(ID=ca.ID).groupby("ID").mean().round(2).to_string())
# cells come from 6 Vgat and 4 Vglut2 mice; the paper treats cells as units. Animal means as units:
am4 = mag.assign(line=ca.Mouse_line.values, ID=ca.ID.values).groupby(["line", "ID"]).mean()
rows = []
for b in BEH:
    x, y = am4.loc["Vgat-Cre"][b], am4.loc["Vglut2-Cre"][b]
    r = stats.ttest_ind(x, y)
    rows.append(dict(behavior=b, vgat_mice=len(x), vglut2_mice=len(y), vgat_mean=x.mean(), vglut2_mean=y.mean(),
                     t=r.statistic, p=r.pvalue, mw_p=stats.mannwhitneyu(x, y).pvalue))
am4t = pd.DataFrame(rows)
am4t["p_bonf"] = mpoa.bonferroni(am4t.p)
print("animal means as units (Student's t, Bonferroni over six; Mann-Whitney uncorrected, smallest possible 2/210 = 0.0095):")
print(am4t.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** Vgat cells have higher magnitudes than Vglut2 cells for ejaculation (2.11 against 0.85, t(791) = 3.998, Bonferroni p = 0.00042), sniff (t = 6.463, p = 1.08e-09) and anogenital sniffing (t = 6.644, p = 3.4e-10), and not for mount (t = 0.337) or intromission (t = -0.320), both p = 1. These are the legend's values. Grooming is t = -7.252 (Vgat lower), as the legend prints and against the Results sentence; it is left out of the outcome. The ejaculation magnitude is recomputed from the long trace; the other five come from the deposited table. With the rescaled 200-frame arrays the same three behaviors differ and mount does not (p = 0.106). The cells come from 6 Vgat and 4 Vglut2 mice. With animal means as units no behavior differs after Bonferroni correction (ejaculation 2.23 against 0.88, t = 1.483, p = 0.176; sniff p = 0.440; anogenital sniffing p = 0.141; grooming p = 0.019, corrected 0.114), so the result rests on treating cells as units, as the paper and the criterion do.

### fig4i-ejaculation-responders

> A larger proportion of Vgat than Vglut2 cells respond positively to ejaculation (184/522
> against 35/271) and to sniffing (29/522 against 1/271).

Responding when the magnitude exceeds 2 (Methods). Chi-square test of proportions (statsmodels
`proportions_chisquare`), Bonferroni over the six behaviors (legend and code). The threshold is
the Methods'; counts at 1.5 and 2.5 are shown for how they move with it.

In [ ]:
rows = []
for b in BEH:
    pos = (mag[b] > 2).values
    chi = proportions_chisquare(count=[pos[vg].sum(), pos[~vg].sum()], nobs=[vg.sum(), (~vg).sum()])
    rows.append(dict(behavior=b, vgat_pos=int(pos[vg].sum()), vglut2_pos=int(pos[~vg].sum()),
                     vgat_frac=pos[vg].mean(), vglut2_frac=pos[~vg].mean(), p=chi[1],
                     vgat_neg=int((mag[b] < -2).values[vg].sum()), vglut2_neg=int((mag[b] < -2).values[~vg].sum()),
                     vgat_pos_1p5=int((mag[b] > 1.5).values[vg].sum()), vglut2_pos_1p5=int((mag[b] > 1.5).values[~vg].sum()),
                     vgat_pos_2p5=int((mag[b] > 2.5).values[vg].sum()), vglut2_pos_2p5=int((mag[b] > 2.5).values[~vg].sum())))
fi = pd.DataFrame(rows)
fi["p_bonf"] = mpoa.bonferroni(fi.p)
print(fi.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("ejaculation responders by animal:")
print(pd.DataFrame({"cells": ca.groupby("ID").size(), "ej_pos": (mag.Ejaculation > 2).groupby(ca.ID).sum(),
                    "sniff_pos": (mag.Sniff > 2).groupby(ca.ID).sum()}).to_string())
fr = pd.DataFrame({"line": ca.Mouse_line.values, "ID": ca.ID.values, "ej": (mag.Ejaculation > 2).values,
                   "sniff": (mag.Sniff > 2).values}).groupby(["line", "ID"]).mean()
print("fraction of cells responding, by animal:"); print(fr.round(3).to_string())
for b in ["ej", "sniff"]:
    x, y = fr.loc["Vgat-Cre"][b], fr.loc["Vglut2-Cre"][b]
    print(f"{b}: animal fractions Vgat mean {x.mean():.3f} (6 mice) against Vglut2 {y.mean():.3f} (4 mice), "
          f"t = {stats.ttest_ind(x, y).statistic:.3f}, p = {stats.ttest_ind(x, y).pvalue:.3g}, Mann-Whitney p = {stats.mannwhitneyu(x, y).pvalue:.3g}")

**Result: reproduced.** 184 of 522 Vgat and 35 of 271 Vglut2 cells respond positively to ejaculation (Bonferroni p = 1.5e-10), and 29 and 1 to sniff (p = 0.0017), the counts and p-values of the legend. All six Vgat and all four Vglut2 animals have ejaculation responders (5 to 55 per animal); 26 of the 29 Vgat sniff responders come from one animal (Vgat_4). Taking animals as units, the fraction of ejaculation responders is 0.373 in Vgat mice against 0.146 in Vglut2 mice (t-test p = 0.104, Mann-Whitney p = 0.038) and the fraction of sniff responders 0.050 against 0.007 (p = 0.46 and 0.694).

### fig5-ejaculation-specific

> Among cells responding to sniff, mount or ejaculation, ejaculation-specific cells are the most
> numerous (177, against 46 mount and 12 sniff), and most ejaculation responders are specific
> (Vgat 80%, Vglut2 82%).

Reading (a): positive (> 2) to that behavior only among sniff, mount and ejaculation, from the
responder flags of fig4i-ejaculation-responders. Reading (b): the table's `magnitude_cell_types`.

In [ ]:
P = {b: (mag[b] > 2).values for b in ["Sniff", "Mount", "Ejaculation"]}
cls = np.full(len(ca), "other", dtype=object)
cls[P["Sniff"] & ~P["Mount"] & ~P["Ejaculation"]] = "sniff_only"
cls[~P["Sniff"] & P["Mount"] & ~P["Ejaculation"]] = "mount_only"
cls[~P["Sniff"] & ~P["Mount"] & P["Ejaculation"]] = "ejaculation_only"
tab_map = {"Sniff_only_cells": "sniff_only", "Mount_only_cells": "mount_only", "Ejaculation_only_cells": "ejaculation_only"}
cls_b = ca.magnitude_cell_types.map(tab_map).fillna("other").values
for lab, c in [("(a) recomputed", cls), ("(b) table column", cls_b)]:
    n = pd.Series(c).value_counts()
    sp_vgat = ((c == "ejaculation_only") & vg).sum() / (P["Ejaculation"] & vg).sum()
    sp_vglut = ((c == "ejaculation_only") & ~vg).sum() / (P["Ejaculation"] & ~vg).sum()
    print(f"{lab}: ejaculation-only {n.get('ejaculation_only', 0)}, mount-only {n.get('mount_only', 0)}, "
          f"sniff-only {n.get('sniff_only', 0)}; specific share of ejaculation responders, Vgat "
          f"{sp_vgat:.3f} ({((c == 'ejaculation_only') & vg).sum()}/{(P['Ejaculation'] & vg).sum()}), Vglut2 "
          f"{sp_vglut:.3f} ({((c == 'ejaculation_only') & ~vg).sum()}/{(P['Ejaculation'] & ~vg).sum()})")
print("(a) and (b) agree for", int((cls == cls_b).sum()), "of", len(ca), "cells")
print("table classes:", ca.magnitude_cell_types.value_counts().to_dict())
print("by line, reading (a):", pd.crosstab(cls, ca.Mouse_line).to_dict())
print("classes of reading (a) by animal:")
print(pd.crosstab([ca.Mouse_line.values, ca.ID.values], cls).to_string())

**Result: reproduced.** Both readings give 178 ejaculation-specific, 48 mount-specific and 12 sniff-specific cells (the legend's 178 and 48; the Results give 177 and 46, both within tolerance). Of the ejaculation responders, 149 of 184 Vgat (0.810) and 29 of 35 Vglut2 (0.829) are specific, against the stated 80% and 82%. The recomputed classes equal the table's for all 793 cells. Every animal has ejaculation-specific cells (2 to 44); the 12 sniff-specific cells come from two animals, Vgat_3 (3) and Vgat_4 (9).

### fig5c-class-selectivity

> Sniff-specific and ejaculation-specific cells respond more strongly than the other classes only
> to sniff and to ejaculation, respectively; mount-specific cells respond more strongly than the
> other classes to mount, intromission and anogenital sniffing, but not to sniff or ejaculation.

Per behavior, one-way ANOVA over the three classes (statsmodels `ols` and `anova_lm`, type 2) and
Tukey's HSD (`pairwise_tukeyhsd`), as the code runs them. Classes of reading (a).

In [ ]:
from statsmodels.formula.api import ols
import statsmodels.api as sm
keep = np.isin(cls, ["sniff_only", "mount_only", "ejaculation_only"])
rows, tk = [], {}
for b in BEH:
    d = pd.DataFrame({"v": mag[b].values[keep], "cls": cls[keep]})
    a = sm.stats.anova_lm(ols("v ~ C(cls)", data=d).fit(), typ=2)
    t = mpoa.tukey(d.v, d.cls)
    tk[b] = t
    means = d.groupby("cls").v.mean()
    rows.append(dict(behavior=b, F=a.F.iloc[0], df=f"({a.df.iloc[0]:.0f},{a.df.iloc[1]:.0f})", p=a["PR(>F)"].iloc[0],
                     **{f"mean_{k}": v for k, v in means.items()}))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
for b in BEH:
    print(b); print(tk[b].to_string(index=False))

**Result: reproduced.** The ANOVA F values are those of the legend (sniff 73.35, mount 68.32, intromission 15.72, anogenital sniffing 12.34, ejaculation 49.59, all on (2,235)). Sniff cells exceed both other classes for sniff (Tukey p < 0.0001), ejaculation cells exceed both for ejaculation (p < 0.0001 and 0.0001), and mount cells exceed both for mount and exceed ejaculation cells for intromission and anogenital sniffing (p < 0.0001). Mount cells do not exceed sniff cells for sniff (0.459 against 2.504) nor ejaculation cells for ejaculation (-0.517 against 5.902).

### fig6c-decay-length

> Ejaculation-responding Vgat cells have a longer decay length than Vglut2 cells.

The -5 to +120 s trace, baseline (-5 to 0 s) subtracted. Readings of the claims file: (a) time
from onset to the first sample below 0, (b) total time below 0 in 0 to 120 s, and (c) time from
the peak after onset to the first sample below 0 after it. Reading (c) was added to the claims
file on a reader's answer (questions/001249.yaml, 2026-11-01), after the first run of this
attempt tested (a) and (b) and showed (c) as a variant. Student's t-test; reproduced when Vgat is
longer at p < 0.05 under a reading.

The paper does not say what value a trace that never falls below 0 takes. The main analysis gives
it the full 120 s window, the value the deposited table holds for every such cell (an
assumption). Two variants are shown: the time from the start of the measure to the end of the
window (the least it can be), and leaving such cells out. The table's precomputed decay length,
whose computation is not in the code, is shown too.

In [ ]:
resp = (mag.Ejaculation > 2).values
tabd_all = ca.Ejaculation_decay_length_seconds.values
rows = []
for lab, dl in [("(a) onset to first sample below 0", mpoa.decay_length(longer, "a")),
                ("(b) total time below 0", mpoa.decay_length(longer, "b")),
                ("(c) peak to first sample below 0", mpoa.decay_length(longer, "c")),
                ("(c) unreturned = peak to end of window", mpoa.decay_length(longer, "c", "to_end")),
                ("(c) unreturned left out", mpoa.decay_length(longer, "c", "drop")),
                ("(a) unreturned left out", mpoa.decay_length(longer, "a", "drop")),
                ("table column", tabd_all)]:
    x, y = dl[resp & vg], dl[resp & ~vg]
    x, y = x[~np.isnan(x)], y[~np.isnan(y)]
    r = stats.ttest_ind(x, y)
    rows.append(dict(reading=lab, n_vgat=len(x), n_vglut2=len(y), vgat_mean=np.mean(x), vglut2_mean=np.mean(y),
                     vgat_median=np.median(x), vglut2_median=np.median(y), diff=np.mean(x) - np.mean(y),
                     t=r.statistic, p=r.pvalue, at_120s=int((x >= 120).sum() + (y >= 120).sum()),
                     u_p=stats.mannwhitneyu(x, y).pvalue))
dc = pd.DataFrame(rows)
print(dc.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
dlc = mpoa.decay_length(longer, "c")
unret_c = np.isnan(mpoa.decay_length(longer, "c", "drop"))
print("responders whose trace never falls below 0 after the peak: Vgat", int((resp & vg & unret_c).sum()),
      "of", int((resp & vg).sum()), "| Vglut2", int((resp & ~vg & unret_c).sum()), "of", int((resp & ~vg).sum()),
      "| table value for them:", sorted(set(tabd_all[resp & unret_c])))
tabd = tabd_all[resp]
for lab in "abc":
    dl = mpoa.decay_length(longer, lab)[resp]
    print(f"reading ({lab}) against the table column: r = {np.corrcoef(dl, tabd)[0, 1]:.3f}, equal (within 0.05 s) in "
          f"{int((abs(dl - tabd) < 0.051).sum())} of {int(resp.sum())} responders")
by = (pd.DataFrame({"line": ca.Mouse_line.values, "ID": ca.ID.values, "c": dlc, "a": mpoa.decay_length(longer, "a"),
                    "unreturned": unret_c})[resp]
      .groupby(["line", "ID"]).agg(n=("c", "size"), unreturned_c=("unreturned", "sum"), mean_c=("c", "mean"),
                                   mean_a=("a", "mean")))
print("by animal (responders):"); print(by.round(1).to_string())
am = by.mean_c
ra = stats.ttest_ind(am[am.index.get_level_values(0) == "Vgat-Cre"], am[am.index.get_level_values(0) == "Vglut2-Cre"])
print(f"reading (c), animal means as units: Vgat {am['Vgat-Cre'].mean():.1f} s (6 mice) against Vglut2 "
      f"{am['Vglut2-Cre'].mean():.1f} s (4 mice), t = {ra.statistic:.3f}, p = {ra.pvalue:.3g}")
x0 = longer - longer[:mpoa.ONSET].mean(0)
peak_s = x0[mpoa.ONSET:].argmax(0) / mpoa.RATE_HZ
dl_a_all = mpoa.decay_length(longer, "a")
dif = resp & (abs(dlc - tabd_all) >= 0.051)
d12 = pd.DataFrame({"ID": ca.ID.values, "line": ca.Mouse_line.values, "peak_s": peak_s,
                    "return_after_peak_s": peak_s + dlc, "c_s": dlc, "a_s": dl_a_all,
                    "b_s": mpoa.decay_length(longer, "b"), "table_s": tabd_all,
                    "table_minus_c": tabd_all - dlc, "unreturned_c": unret_c})[dif]
print(f"responders where reading (c) and the table differ by 0.05 s or more: {int(dif.sum())}")
print(d12.round(2).to_string())

**Result: reproduced, resting on an assumption.** Under reading (c), added to the claims file on a reader's answer, Vgat responders take longer from the peak to fall below 0: 45.2 against 20.9 s, t = 3.037, p = 0.0027, close to the paper's t(217) = 3.040, p = 0.00266, and (c) equals the deposited decay lengths for 207 of 219 cells (r = 1.000). This rests on giving the 44 cells that never fall below 0 after the peak (43 Vgat, 1 Vglut2, 31 of them in mouse Vgat_2) the full 120 s, as the deposited table does for all 44. Counting them as the time from the peak to the end of the window gives 28.9 against 20.6 s (p = 0.111), and leaving them out gives 22.4 against 18.0 s (p = 0.299); the rank-sum test under (c) gives p = 0.187, and the animal means give 40.9 against 20.2 s (6 and 4 mice, p = 0.149). Under readings (a) and (b) Vgat cells do not decay later: (a) 18.7 against 22.2 s (t = -0.608, p = 0.544), (b) 53.9 against 75.3 s (t = -2.641, p = 0.0089, Vgat shorter). Reading (c) and the deposited table differ for 12 of the 219 responders, 11 of them in Vgat_4: none is a cell that never returns below 0, and the table is longer by 0.3 to 5.2 s in each, so the table's computation differs from (c) in a detail that the outputs do not identify; the table column gives the same test (t = 3.04, p = 0.00266).

### fig6k-late-cluster-vgat

> Spectral clustering of ejaculation responses gives five types; the late-response type 3 has
> the longest decay length and contains only Vgat cells, while Vglut2 cells are more often fast
> responders.

As the claims file gives it (Methods; Figure 5 Spectral clustering.ipynb): the -5 to +120 s
traces of the 219 ejaculation responders, whitened PCA, the number of components at the scree
elbow (the Methods state 4), spectral clustering with nearest-neighbour affinity over 2 to 11
clusters and 10 to 70 neighbours, the setting with the largest cosine silhouette, and clusters
numbered by their 0 to 5 s response, largest first (the code's reorder_clusters). The code
fixes no seed; seed 0 is a standard choice, and seeds 1 to 4 are shown for stability. The late
cluster is the one whose mean trace peaks latest. The fast cluster is type 1: the code numbers
the clusters by their mean 0 to 5 s response, largest first (reorder_clusters in Figure 5
Spectral clustering.ipynb), and the paper's type numbers are the code's; the cluster whose mean
trace peaks earliest is shown as a variant. The number of components follows the code's elbow
rule, which keeps the components before the elbow (np.argmin of the rotated scree plot, used as a
count). Proportions by chi-square test (`proportions_chisquare`), Bonferroni over the
clusters. The paper's setting (5 clusters, 55 neighbours) and the deposited labels are shown too.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
t = time.time()
X = longer[:, resp].T
vg_r = vg[resp]
pca = PCA(n_components=min(X.shape), whiten=True, svd_solver="full")
Z = pca.fit_transform(X)
n_pc = mpoa.elbow_pcs(pca.explained_variance_ratio_)
print("cells", X.shape[0], "| PCs kept by the code's elbow rule (0-based index of the elbow, used as a count):",
      n_pc, "(Methods: 4)")
Z4 = Z[:, :4]
dec_c = mpoa.decay_length(longer, "c")[resp]
dec_a = mpoa.decay_length(longer, "a")[resp]


def describe(labels, label):
    lab = mpoa.order_clusters(labels, X.T)
    rows = []
    n_vgat, n_vglut = vg_r.sum(), (~vg_r).sum()
    for c in np.unique(lab):
        m = lab == c
        tr = X[m].mean(0)
        chi = proportions_chisquare([int((m & vg_r).sum()), int((m & ~vg_r).sum())], [n_vgat, n_vglut])
        rows.append(dict(cluster=c + 1, n=int(m.sum()), vgat=int((m & vg_r).sum()), vglut2=int((m & ~vg_r).sum()),
                         vgat_frac=(m & vg_r).sum() / n_vgat, vglut2_frac=(m & ~vg_r).sum() / n_vglut,
                         mean_0_5s=tr[50:100].mean(), peak_time_s=(tr[50:].argmax()) / 10,
                         mean_60_120s=tr[650:].mean(), decay_c_mean=dec_c[m].mean(), decay_a_mean=dec_a[m].mean(), p=chi[1]))
    out = pd.DataFrame(rows)
    out["p_bonf"] = mpoa.bonferroni(out.p)
    late = int(out.loc[out.peak_time_s.idxmax(), "cluster"])
    print(f"{label}: late cluster (latest peak) = {late}; cluster with the longest decay (c) = "
          f"{int(out.loc[out.decay_c_mean.idxmax(), 'cluster'])}, (a) = {int(out.loc[out.decay_a_mean.idxmax(), 'cluster'])}")
    print(out.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    return lab, out, late


grids = {}
for seed in range(5):
    g = mpoa.spectral_grid(Z4, seed=seed)
    grids[seed] = g
    b = g.loc[g.silhouette.idxmax()]
    print(f"seed {seed}: best {b.n_clusters} clusters, {b.n_neighbors} neighbours, silhouette {b.silhouette:.3f}; "
          f"5 clusters at 55 neighbours {g[(g.n_clusters == 5) & (g.n_neighbors == 55)].silhouette.iloc[0]:.3f}")
print(f"grid time {time.time() - t:.0f} s")
g0 = grids[0]
print("best silhouette per number of clusters (seed 0):",
      g0.groupby("n_clusters").silhouette.max().round(3).to_dict())
best = g0.loc[g0.silhouette.idxmax()]
lab_main, k_main, late_main = describe(best.labels, f"main, seed 0, {best.n_clusters} clusters, {best.n_neighbors} neighbours")
for seed in range(1, 5):
    bs = grids[seed].loc[grids[seed].silhouette.idxmax()]
    print(f"seed {seed}: agreement with seed 0 (adjusted Rand) {adjusted_rand_score(best.labels, bs.labels):.3f}")
p55 = g0[(g0.n_clusters == 5) & (g0.n_neighbors == 55)].iloc[0]
lab55, k55, late55 = describe(p55.labels, "paper's setting, 5 clusters, 55 neighbours")
dep_lab = mpoa.read_npy(F4, "clustering/202308071543_spectral_clusterlabels.npy")
print("deposited labels:", dep_lab.shape, "| agreement with main (adjusted Rand)",
      round(adjusted_rand_score(dep_lab, best.labels), 3), "| with the 55-neighbour fit", round(adjusted_rand_score(dep_lab, p55.labels), 3))
print("deposited labels (the code numbers them by 0-5 s response and swaps two, so numbers differ):")
lab_dep, k_dep, late_dep = describe(dep_lab, "deposited labels")
# The fast cluster is type 1, the cluster with the largest 0 to 5 s response, as the code's
# reorder_clusters numbers them; the cluster whose mean trace peaks earliest is shown as a variant.
for lab_k, k in [("main", k_main), ("55 neighbours", k55), ("deposited", k_dep)]:
    f1 = k.loc[k.mean_0_5s.idxmax()]
    fe = k.loc[k.peak_time_s.idxmin()]
    print(f"{lab_k}: largest 0-5 s response, cluster {int(f1.cluster)} (peak {f1.peak_time_s:.1f} s): Vgat {int(f1.vgat)}/184, "
          f"Vglut2 {int(f1.vglut2)}/35, p_bonf {f1.p_bonf:.3g} | earliest peak, cluster {int(fe.cluster)} (peak {fe.peak_time_s:.1f} s): "
          f"Vgat {int(fe.vgat)}/184, Vglut2 {int(fe.vglut2)}/35, p_bonf {fe.p_bonf:.3g}")
print("main clusters by animal:")
print(pd.crosstab([ca.Mouse_line.values[resp], ca.ID.values[resp]], lab_main + 1).to_string())

**Result: reproduced.** The scree elbow keeps 4 components and the silhouette is largest at 5 clusters (0.474, at 30 neighbours), for all five seeds, with labels equal to the deposited ones (adjusted Rand 1.0). The late cluster (peak at 48.5 s, longest decay under readings a and c) holds 37 of 184 Vgat and 0 of 35 Vglut2 cells (Bonferroni p = 0.0181), and the fast cluster holds 16 of 35 Vglut2 against 26 of 184 Vgat cells (p = 6.8e-05). With the paper's stated 55 neighbours (silhouette 0.461, not the stated 0.405) the late cluster holds 33 Vgat and 0 Vglut2 cells (p = 0.0328). The elbow rule gives 4, as the code printed it. The late cluster's 37 Vgat cells come from 5 of the 6 Vgat mice, 27 of them from Vgat_2. The fast part holds for type 1 as the code numbers it; the cluster whose mean trace peaks earliest (1.9 s, against 5.8 s for type 1) holds 41 of 184 Vgat and 5 of 35 Vglut2 cells (Bonferroni p = 1). That earliest-peak cluster is not a candidate for the fast type: its 46 cells (41 Vgat, 5 Vglut2) are the paper's Type 5, which the Results call the negative response type, while type 1 has the 42 cells (26 Vgat, 16 Vglut2) the Figure 6 legend gives for Type 1, the type the Results call fast response. At 55 neighbours the late cluster's 33 Vgat cells are outside the tolerance of 37 +/- 3.7; the outcome rests on the 30 neighbours that the claims file's rule (largest silhouette) selects.

## Figure 7 and its supplements: TRAP of MPOA ensembles and chemogenetics (Figshare)

In [ ]:
F7 = "Figure 6"
g7 = mpoa.read_csv(F7, "data/group_df.csv")
print("Figure 7 table: rows by Experiment_type", g7.Experiment_type.value_counts().to_dict())
si7 = g7[g7.Experiment_type == "SI"]
print("self-paced (SI) animals by group and drug:")
print(si7.groupby(["Stim_Group", "drug"]).ID.nunique().unstack().to_string())
print("one row per animal and drug:", bool((si7.groupby(["ID", "drug"]).size() == 1).all()))
rs = si7.Intromission_event / si7.Mount_event.replace(0, np.nan)
print("Receptive_score against Intromission_event / Mount_event (legend's definition): equal in",
      int((abs(rs - si7.Receptive_score) < 1e-9).sum()), "of", int(rs.notna().sum()), "rows with mounts;",
      "Receptive_score is 0 in", int((si7.Receptive_score[si7.Mount_event == 0] == 0).sum()), "of",
      int((si7.Mount_event == 0).sum()), "rows without mounts")
print("Duration_second against Average_Isolation_zone_duration: equal in",
      int((abs(si7.Duration_second - si7.Average_Isolation_zone_duration) < 1e-6).sum()), "of", len(si7),
      "| against Average_Appetitive_zone_duration:", int((abs(si7.Duration_second - si7.Average_Appetitive_zone_duration) < 1e-6).sum()))
print("Average_Isolation_zone_duration / Duration_second by group (median):",
      (si7.Average_Isolation_zone_duration / si7.Duration_second).groupby(si7.Stim_Group).median().round(2).to_dict())
print("ranges:", si7[["Number_of_zone_transition_per_10min", "Duration_second", "Average_Isolation_zone_duration",
                      "Intromission_event", "Mount_event", "Grooming_event", "Receptive_score"]].agg(["min", "max"]).round(2).to_dict())
trap = mpoa.read_csv("Figure S9", "data/FigS9_all_TRAP_cells.csv")
ish = mpoa.read_csv("Figure S9", "data/FigS9_ISH_data.csv")
print("TRAP counts: mice by Stim", trap.Stim.value_counts().to_dict(), "| by Group_Stim", trap.Group_Stim.value_counts().to_dict())
print("ISH: mice by Stim_Group", ish.Stim_Group.value_counts().to_dict())
has = trap.Probe_cells > 0
print("Proportion_of_TRAP_cells_in_MPN = MPN_probe_cells / Probe_cells where Probe_cells > 0:",
      bool(np.allclose(trap.Proportion_of_TRAP_cells_in_MPN[has], (trap.MPN_probe_cells / trap.Probe_cells)[has])),
      "| mice with no TRAP cells:", trap[~has][["ID", "Group_Stim", "Proportion_of_TRAP_cells_in_MPN"]].to_dict("records"))
print("TRAP cells range", trap.Probe_cells.min(), "to", trap.Probe_cells.max())
print("Vgat_probability = Vgat_TRAP_cells / TRAP_cells:", bool(np.allclose(ish.Vgat_probability, ish.Vgat_TRAP_cells / ish.TRAP_cells)),
      "| Vglut2:", bool(np.allclose(ish.Vglut2_probability, ish.Vglut2_TRAP_cells / ish.TRAP_cells)))
hc = mpoa.read_csv("Figure S10", "data/homecage_summary_df.csv")
cpp = mpoa.read_csv("Figure S10", "data/CPP_behaviordf.csv")
print("home cage (activation): mice by Stim_Group", hc.Stim_Group.value_counts().to_dict(), "| State", hc.State.value_counts().to_dict())
print("CPP: rows by Session", cpp.Session.value_counts().to_dict(), "| mice by Stim_Group", cpp.groupby("Stim_Group").ID.nunique().to_dict())
g11 = mpoa.read_csv("Figure S11", "data/group_df.csv")
print("inhibition table: animals by Experiment_type, Group, drug")
print(g11.groupby(["Experiment_type", "Group", "drug"]).ID.nunique().unstack().to_string())

### fig7fg-activation-suppresses

> CNO activation of post-ejaculatory TRAP-labeled MPOA neurons reduces zone transitions and
> increases time in the isolation zone; activation of appetitive or consummatory ensembles and
> eYFP controls do not.

Saline against CNO trial within each group, Bonferroni over the four groups. Main reading: the
legend's Wilcoxon rank-sum test (scipy `mannwhitneyu`, two-sided, here equal to the rank-sum
test); variant: the code's paired signed-rank test (scipy `wilcoxon`), matched by animal. Time in
the isolation zone: the code tests `Duration_second`; the legend's "average time in isolation
zone" is `Average_Isolation_zone_duration`. Both columns are run (data check above).

In [ ]:
GROUPS7 = ["Post_ejaculatory_hM3Dq", "Post_ejaculatory_eYFP", "Appetitive_hM3Dq", "Consummatory_hM3Dq"]


def saline_cno(df, groups, cols):
    rows = []
    for name, col in cols:
        rs_p, sr_p = [], []
        for gname in groups:
            sal, cno, ids = mpoa.paired(df[df.Stim_Group == gname], "ID", "drug", "Saline", "CNO", col)
            rs_p.append(stats.mannwhitneyu(sal, cno).pvalue)
            sr_p.append(stats.wilcoxon(sal, cno, method="approx").pvalue if np.any(sal != cno) else 1.0)
            rows.append(dict(measure=name, group=gname, n=len(ids), saline_mean=sal.mean(), cno_mean=cno.mean(),
                             n_cno_lower=int((cno < sal).sum()), n_cno_higher=int((cno > sal).sum()),
                             p_ranksum=rs_p[-1], p_signedrank=sr_p[-1],
                             p_signedrank_exact=stats.wilcoxon(sal, cno, method="exact").pvalue if np.any(sal != cno) else 1.0))
        k = len(groups)
        for r in rows[-k:]:
            r["p_ranksum_bonf"] = min(r["p_ranksum"] * k, 1)
            r["p_signedrank_bonf"] = min(r["p_signedrank"] * k, 1)
            r["p_signedrank_exact_bonf"] = min(r["p_signedrank_exact"] * k, 1)
    return pd.DataFrame(rows)


fg = saline_cno(si7, GROUPS7, [("transitions per 10 min", "Number_of_zone_transition_per_10min"),
                               ("isolation time, Duration_second (code)", "Duration_second"),
                               ("isolation time, Average_Isolation_zone_duration (legend)", "Average_Isolation_zone_duration")])
print(fg.drop(columns=["p_ranksum", "p_signedrank", "p_signedrank_exact"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

**Result: partially reproduced.** In post-ejaculatory hM3Dq mice CNO lowers transitions (2.28 to 0.14 per 10 min, all 8 lower) and raises isolation time (all 8 higher); by the main reading, the rank-sum test, Bonferroni p = 0.0076 and 0.0025. By the same test the consummatory hM3Dq group also has fewer transitions (2.13 to 0.31, 7 of 7 lower, p = 0.0157), so the claim that only the post-ejaculatory group changes fails for transitions. Isolation time changes only in the post-ejaculatory group (others 0.163 to 1). The code's signed-rank test (variant) gives the legend's values: post-ejaculatory 0.03125 for both measures, consummatory 0.0625 and 0.125. Whether the signed-rank test should be the main reading, as the legend's p-values and the paired design suggest, is asked of a reader (questions/001249.yaml); the outcome is the one the claims file's main reading gives.

### fig7kl-activation-intromission

> Activation of post-ejaculatory TRAP neurons lowers intromissions and receptivity in the
> self-paced assay; the text also reports fewer self-grooming and mounting episodes, which the
> legend gives as p = 0.0625.

In [ ]:
kl = saline_cno(si7, GROUPS7, [("intromission", "Intromission_event"), ("receptive score", "Receptive_score"),
                               ("self-grooming", "Grooming_event"), ("mount", "Mount_event"), ("sniff", "Sniff_event")])
print(kl.drop(columns=["p_ranksum", "p_signedrank", "p_signedrank_exact"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** In post-ejaculatory hM3Dq mice CNO lowers intromission (5.5 to 0.25) and the receptive score (0.503 to 0.031), all 8 lower, rank-sum Bonferroni p = 0.0026 and 0.0033. Self-grooming (23.5 to 2.25) and mounting (18.0 to 1.25) are lower too, p = 0.0103 and 0.0093 by the rank-sum test, and 0.0625 by the code's exact signed-rank test, the legend's value.

### fig7s1-mpn-proportion

> The number of TRAP-labeled cells does not differ between groups, but the proportion located in
> the MPN is higher in the post-ejaculatory group; most MPN TRAP cells are Vgat (0.76) and few
> Vglut2 (0.15).

One-way ANOVA (`ols`, `anova_lm` type 2) over Stim, Tukey HSD (Figure S9.ipynb), with the
post-ejaculatory eYFP and hM3Dq mice pooled (legend B and C). Number of TRAP cells is the
`Probe_cells` column the code tests. Vgat against Vglut2 proportion within each group: unpaired
Student's t-test (main reading of the claims file) and the code's paired test (variant). Means
over (a) both groups pooled, (b) post-ejaculatory mice alone.

In [ ]:
for col in ["Probe_cells", "Proportion_of_TRAP_cells_in_MPN"]:
    a = sm.stats.anova_lm(ols(f"{col} ~ C(Stim)", data=trap).fit(), typ=2)
    print(f"{col}: F({a.df.iloc[0]:.0f},{a.df.iloc[1]:.0f}) = {a.F.iloc[0]:.4f}, p = {a['PR(>F)'].iloc[0]:.4g};",
          "means", trap.groupby("Stim")[col].mean().round(4).to_dict(), "| n", trap.Stim.value_counts().to_dict())
    print(mpoa.tukey(trap[col], trap.Stim).to_string(index=False))
rows = []
for stim, d in ish.groupby("Stim"):
    tu = stats.ttest_ind(d.Vgat_probability, d.Vglut2_probability)
    tr = stats.ttest_rel(d.Vgat_probability, d.Vglut2_probability)
    rows.append(dict(group=stim, n=len(d), vgat_mean=d.Vgat_probability.mean(), vgat_sd=d.Vgat_probability.std(),
                     vglut2_mean=d.Vglut2_probability.mean(), vglut2_sd=d.Vglut2_probability.std(),
                     t_unpaired=tu.statistic, df_unpaired=2 * len(d) - 2, p_unpaired=tu.pvalue,
                     t_paired=tr.statistic, p_paired=tr.pvalue, p_paired_x2=min(2 * tr.pvalue, 1)))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
post_ish = ish[ish.Stim == "Post_ejaculatory"]
print(f"(a) pooled, n={len(ish)}: Vgat {ish.Vgat_probability.mean():.3f} +/- {ish.Vgat_probability.std():.3f}, "
      f"Vglut2 {ish.Vglut2_probability.mean():.3f} +/- {ish.Vglut2_probability.std():.3f}")
print(f"(b) post-ejaculatory, n={len(post_ish)}: Vgat {post_ish.Vgat_probability.mean():.3f} +/- {post_ish.Vgat_probability.std():.3f}, "
      f"Vglut2 {post_ish.Vglut2_probability.mean():.3f} +/- {post_ish.Vglut2_probability.std():.3f}")

**Result: reproduced.** The number of TRAP cells does not differ (F(2,24) = 0.3072, p = 0.7383), the proportion in the MPN does (F(2,24) = 4.3335, p = 0.0247; Tukey appetitive against post-ejaculatory p = 0.0471), as in the legend. The Vgat proportion exceeds the Vglut2 proportion in both groups (unpaired t(10) = 7.209, p = 2.9e-05; t(12) = 12.94, p = 2.1e-08). The proportions are 0.743 and 0.124 pooled over 13 mice and 0.786 and 0.138 in the post-ejaculatory mice, both within the stated 0.76 +/- 0.16 and 0.15 +/- 0.13. The legend's t values are those of the paired test (6.615, 13.95) with its p doubled. Whether the paired test should be the main reading is asked of a reader (questions/001249.yaml); both tests are significant.

### fig7s2-home-cage-activation

> In the home cage, activation of post-ejaculatory TRAP neurons reduces sniffing and intromission
> compared with post-ejaculatory eYFP controls, without changing mounting; it does not induce
> place preference or aversion.

Home cage: each hM3Dq group against post-ejaculatory eYFP, two-sided Mann-Whitney U, Bonferroni
over the three comparisons (Figure S10.ipynb; the legend gives no number). Place preference:
post-test minus pre-test in the CNO-paired zone, Mann-Whitney U for eYFP against each hM3Dq group
and between them, Bonferroni over three (Figure S10.ipynb).

In [ ]:
ctrl = hc[hc.Stim_Group == "Post_ejaculatory_eYFP"]
rows = []
for col in ["Sniff_event", "Intromission_event", "Mount_event", "Receptive_score"]:
    ps = []
    for gname in ["Appetitive_hM3Dq", "Consummatory_hM3Dq", "Post_ejaculatory_hM3Dq"]:
        x = hc[hc.Stim_Group == gname][col]
        ps.append(stats.mannwhitneyu(ctrl[col], x).pvalue)
        rows.append(dict(measure=col, group=gname, n_eyfp=len(ctrl), n=len(x), eyfp_mean=ctrl[col].mean(), mean=x.mean(), p=ps[-1]))
    for r, pb in zip(rows[-3:], mpoa.bonferroni(ps)):
        r["p_bonf"] = pb
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
meta_cols = ["ID", "Clean_zone", "Conditioned", "State", "Stim", "Group", "Stim_Group", "CNO_side"]
num = ["Zone_Cumulative Duration", "Zone_Frequency", "Velocity"]
pre = cpp[cpp.Session == "Pre-test"].set_index(meta_cols[:3] + ["Stim_Group"])[num]
postt = cpp[cpp.Session == "Post-test"].set_index(meta_cols[:3] + ["Stim_Group"])[num]
delta = (postt - pre).reset_index()
delta = delta[delta.Conditioned.astype(bool)]
print("CPP delta rows (CNO-paired zone):", len(delta), delta.Stim_Group.value_counts().to_dict())
rows = []
for col in num:
    pairs = [("Post_ejaculatory_eYFP", "Appetitive_hM3Dq"), ("Post_ejaculatory_eYFP", "Post_ejaculatory_hM3Dq"),
             ("Post_ejaculatory_hM3Dq", "Appetitive_hM3Dq")]
    ps = [stats.mannwhitneyu(delta[delta.Stim_Group == a][col], delta[delta.Stim_Group == b][col]).pvalue for a, b in pairs]
    for (a, b), p, pb in zip(pairs, ps, mpoa.bonferroni(ps)):
        rows.append(dict(measure=col, a=a, b=b, mean_a=delta[delta.Stim_Group == a][col].mean(),
                         mean_b=delta[delta.Stim_Group == b][col].mean(), p=p, p_bonf=pb))
# the Bonferroni p-values the legend of Figure 7-figure supplement 2H to 2J prints, as reference
legend_cpp = {"Zone_Cumulative Duration": [0.2203, 1, 1], "Zone_Frequency": [1, 1, 1], "Velocity": [1, 0.8472, 0.8472]}
cppt = pd.DataFrame(rows)
cppt["legend_p_bonf"] = [v for col in num for v in legend_cpp[col]]
print(cppt.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

**Result: reproduced.** Against post-ejaculatory eYFP, only the post-ejaculatory hM3Dq group sniffs less (4.4 against 27.3 events, Bonferroni p = 0.0072) and has fewer intromissions (1.4 against 32.2, p = 0.0165); mounting does not differ (p = 0.4116). These are the legend's values. Time and entries in the CNO-paired zone do not differ between groups (smallest Bonferroni p = 0.162), nor does velocity (smallest 0.0615), so there is no place preference or aversion by this test. For place preference, the time and entries against the eYFP group and the velocity against the eYFP group equal the legend's Bonferroni p-values (0.2203, 1, 1, 1, 1, 0.8472); for post-ejaculatory hM3Dq against appetitive hM3Dq the legend prints 1, 1 and 0.8472 where this test gives 0.162, 0.696 and 0.0615, none significant either way.

### fig7s3-inhibition

> Chemogenetic inhibition of post-ejaculatory TRAP neurons does not change behavior in the
> self-paced assay, but increases intromission and receptivity in the home cage assay.

Home cage (CNO), hM4Di against eYFP, two-sided Mann-Whitney U without correction (legend O to R,
Figure S11.ipynb). Self-paced assay, saline against CNO within each group, Bonferroni over the two
groups; rank-sum test main, signed-rank variant, as in fig7fg-activation-suppresses.

In [ ]:
hc11 = g11[(g11.Experiment_type.str.lower() == "homecage") & (g11.drug == "CNO")]
rows = []
for col in ["Sniff_event", "Mount_event", "Intromission_event", "Receptive_score"]:
    e = hc11[hc11.Group == "eYFP"][col]
    h4 = hc11[hc11.Group == "hM4Di"][col]
    rows.append(dict(measure=col, n_eyfp=len(e), n_hM4Di=len(h4), eyfp_mean=e.mean(), hM4Di_mean=h4.mean(),
                     p=stats.mannwhitneyu(e, h4).pvalue))
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
si11 = g11[g11.Experiment_type == "SI"]
s3 = saline_cno(si11, ["Post_ejaculatory_eYFP", "Post_ejaculatory_hM4Di"],
                [("transitions per 10 min", "Number_of_zone_transition_per_10min"),
                 ("average isolation visit", "Average_Isolation_zone_duration"),
                 ("average interaction visit", "Average_Interaction_zone_duration"),
                 ("sniff", "Sniff_event"), ("self-grooming", "Grooming_event"), ("mount", "Mount_event"),
                 ("intromission", "Intromission_event"), ("receptive score", "Receptive_score")])
print(s3.drop(columns=["p_ranksum", "p_signedrank", "p_signedrank_exact"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("smallest corrected p in the self-paced assay: rank-sum", round(s3.p_ranksum_bonf.min(), 4),
      "| signed-rank", round(s3.p_signedrank_bonf.min(), 4))

**Result: reproduced.** In the home cage, hM4Di mice have more intromissions (77.0 against 50.5, p = 0.0045) and a higher receptive score (0.836 against 0.653, p = 0.0200) than eYFP mice, and sniffing (p = 0.697) and mounting (p = 0.400) do not differ, as in the legend. No self-paced measure changes with CNO in either group (smallest Bonferroni p 0.117 by rank-sum, 0.100 by signed-rank).

## Figure 2: whole-brain TRAP mapping (dandiset), blocked

The seven Figure 2 claims (fig2-density-link, fig2c-brain-wide-increase,
fig2ei-social-regions-higher, fig2-mpn-higher, fig2f-mpn-fourth-largest,
fig2gh-consummatory-tendency, fig2s-dorsal-raphe) need tdTomato+ cell densities per atlas region
and brain. The dandiset holds only the raw volumes (data check): no registration, segmentation,
cell counts, atlas labels or region volumes. Computing the densities means streaming the volumes
(the TRAP channel for the cells and the background channel for the registration), registering
each brain to the Unified Anatomical Atlas with ClearMap, and segmenting cells with an ilastik
classifier that is not deposited and that the Methods trained on manually labelled chunks
("To train the classifier, 10 chunks of images from the four of the subjects were randomly
sampled and used as the training", Methods, Brain-wide analysis). The data check printed the
streaming rate and the time to read the volumes once; that time alone exceeds the 60 minute
limit of a notebook and the time left in this run, before any registration or segmentation.
The 60 minute limit is the registry's: CI fails a notebook that runs over sixty minutes
(docs/protocol.md, Author, step 3: "Aim for under ten minutes; CI fails a notebook over sixty"; and "Develop on a subset": "If the estimate is over the notebook limit, or over the time left ..., record the claim as blocked, `too_large`").

What can be checked without the densities is checked below: the Figure 2 zip on Figshare (its
members, its subject table and the density table's Stim labels), the matching of the 14 brains
to that table, and the cost of the cheaper reads a later attempt might use (a band of rows
around a region, and a plane subsample for registration). Every Figure 2 claim corrects over a
family of atlas regions that covers the brain (Benjamini-Hochberg over the level-4 or level-6
regions; the rank of the MPN among all regions), so each needs every region of every brain
segmented at full resolution; reading only the regions named would not test them. The claims
are blocked as `too_large`; no registration, segmentation or density was computed.

In [ ]:
F2 = "Figure 2 and S3, S4"
print("members of the Figure 2 zip (MB):")
for i in mpoa.figshare_zip(F2).infolist():
    if not i.filename.endswith("/"):
        print(f"  {i.filename.split('/', 1)[1]:70s} {i.file_size / 1e6:9.1f}")
print(mpoa.read_member(F2, "data/memo.txt").decode())
meta2 = mpoa.read_csv(F2, "meta/metainfo.csv")
td = mpoa.read_csv(F2, "data/tdTomato_summary.csv")
print("density table:", td.shape, "| mice by Stim:", td.groupby("Stim").ID.nunique().to_dict())
print("density = newcounts / size:", bool(np.allclose(td.density, td.newcounts / td["size"].replace(0, np.nan), equal_nan=True)))
# link of the 14 brains to the table: the dandiset's subject (sub-<Group>-<n>) against the table's experiment_id (<Group>_<n>)
dsub = pd.DataFrame({"dandi_subject": assets.path.str.extract(r"sub-([\w-]+?)/")[0].values,
                     "nwb_subject_id": nwb.subject.values, "session_start": nwb.session_start.values,
                     "planes": [s[0] for s in nwb["shape"]]})
meta2["dandi_subject"] = meta2.experiment_id.str.replace("_", "-")
lk = meta2[["experiment_id", "ID", "Stim", "Folder_name", "Anterior_top", "Ventral_first", "dandi_subject"]].merge(
    dsub, on="dandi_subject", how="outer", indicator=True)
lk["table_Stim"] = lk.ID.map(td.groupby("ID").Stim.first())
print(lk.drop(columns=["dandi_subject"]).to_string(index=False))
print("matched:", int((lk._merge == "both").sum()), "of", len(dsub), "brains | Stim of metainfo equals the density table's:",
      bool((lk.Stim == lk.table_Stim).all()))
print("dandiset groups against the table's Stim:", pd.crosstab(lk.dandi_subject.str.split("-").str[0], lk.Stim).to_dict())

# cost of cheaper reads on one brain (the volumes are contiguous, so a full-width band of rows in a
# plane is one range read, and a crop of a plane is one read per row)
f = mpoa.open_nwb(assets.s3_url.iloc[0])
d = f["acquisition/TRAP/data"]
z = d.shape[0] // 2
y0 = d.shape[1] // 2 - 256
t = time.time(); band = d[z + 1:z + 4, y0:y0 + 512, :]; t_band = time.time() - t
x0 = d.shape[2] // 2 - 256
t = time.time(); crop = d[z + 5:z + 8, y0:y0 + 512, x0:x0 + 512]; t_crop = time.time() - t
f.close()
print(f"{assets.path.iloc[0].split('/')[0]}: 3 bands of 512 full rows ({band.nbytes / 1e6:.1f} MB) in {t_band:.2f} s "
      f"({band.nbytes / 1e6 / t_band:.1f} MB/s); 3 crops of 512 x 512 ({crop.nbytes / 1e6:.1f} MB) in {t_crop:.2f} s "
      f"({crop.nbytes / 1e6 / t_crop:.1f} MB/s)")
planes_all = sum(s[0] for s in nwb["shape"])
plane_mb = np.mean([s[1] * s[2] * 2 / 1e6 for s in nwb["shape"]])
for step in [10, 25]:
    n = planes_all // step
    print(f"every {step}th background plane of all 14 brains, for a registration at {4 * step} um in z: {n} planes, "
          f"{n * plane_mb / 1e3:.1f} GB, {n * plane_mb / rate / 60:.0f} min at {rate:.1f} MB/s (registration itself not run)")

In [ ]:
print(f"TRAP channels {vox:.0f} GB, both channels {2 * vox:.0f} GB; at {rate:.1f} MB/s, "
      f"{vox * 1e3 / rate / 60:.0f} min to stream the TRAP channels and {2 * vox * 1e3 / rate / 60:.0f} min for both")
print(f"notebook run time so far: {(time.time() - T0) / 60:.1f} min")

**Result: blocked (too large), all seven Figure 2 claims.** The 14 brains match the 14 rows of the Figure 2 zip's subject table by name (Sniff_n, Mount_n, Ejaculation_n against sub-Sniff-n and so on), and the groups (4 Sniff, 5 Mount, 5 Ejaculation) equal the density table's Stim labels (4 Appetitive, 5 Consummatory, 5 Post_ejaculatory), so the grouping part of fig2-density-link holds; its density part needs the densities, which were not computed. The subject table also gives each brain's orientation (Anterior_top, Ventral_first), which the NWB files do not. The zip's memo calls the region size an area in mm2 and the density cells per mm2. Reading only a band of rows around a region is possible (full-width bands streamed at 20.4 MB/s in this run, crops at 2.5 MB/s), and a plane subsample of the background channels for registration takes 7 to 17 min at this run's 67.4 MB/s, but every Figure 2 claim needs all regions of every brain segmented at full resolution, which means streaming the TRAP channels (171 min at 67.4 MB/s in this run, and 5.4 h at the 35.5 MB/s of an earlier run; the rate varies between runs), and the segmentation classifier is not deposited.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig2-density-link | dandiset | blocked, too large |
| fig2c-brain-wide-increase | dandiset | blocked, too large |
| fig2ei-social-regions-higher | dandiset | blocked, too large |
| fig2-mpn-higher | dandiset | blocked, too large |
| fig2f-mpn-fourth-largest | dandiset | blocked, too large |
| fig2gh-consummatory-tendency | dandiset | blocked, too large |
| fig2s-dorsal-raphe | dandiset | blocked, too large |
| fig1gh-transitions-isolation | Figshare | reproduced |
| fig1i-latency-return | Figshare | reproduced |
| fig1jm-behaviors-after-ejaculation | Figshare | reproduced |
| fig1nt-natural-estrus | Figshare | reproduced |
| fig1s1-experience | Figshare | reproduced |
| fig1s2-novel-male | Figshare | reproduced |
| fig3d-mpn-fos | Figshare | reproduced |
| fig3i-vgat-vglut2-fos | Figshare | reproduced |
| fig3jm-cell-types | Figshare | reproduced |
| fig4h-response-magnitude | Figshare | reproduced |
| fig4i-ejaculation-responders | Figshare | reproduced |
| fig5-ejaculation-specific | Figshare | reproduced |
| fig5c-class-selectivity | Figshare | reproduced |
| fig6c-decay-length | Figshare | reproduced under reading (c), from the peak, added on a reader's answer; rests on giving cells that never return below 0 the full 120 s |
| fig6k-late-cluster-vgat | Figshare | reproduced |
| fig7fg-activation-suppresses | Figshare | partially reproduced (rank-sum test; the code's signed-rank test reproduces it) |
| fig7kl-activation-intromission | Figshare | reproduced |
| fig7s1-mpn-proportion | Figshare | reproduced |
| fig7s2-home-cage-activation | Figshare | reproduced |
| fig7s3-inhibition | Figshare | reproduced |

How the analysis differs from the paper's. The Figure 2 claims, the only ones on the
dandiset, were not run: the dandiset holds raw light-sheet volumes only, and the registration
and segmentation would need the volumes streamed (hours at the measured rate) and an ilastik
classifier that is not deposited. All other claims were tested on the authors' Figshare deposit,
mostly on per-animal or per-cell tables the paper's code reads; the HCR proportions were
recomputed from the per-cell table, the ejaculation magnitudes and decay lengths from the long
traces, and the clustering was run again. Figures 4 to 6 treat cells as units, as the paper does; with animal means as units the magnitude differences of Figure 4H are not significant, the ejaculation responder fraction is significant by Mann-Whitney only, and the Figure 6C decay length is not significant. The response magnitudes for the five behaviors other
than ejaculation come from the deposited table, because the traces they are computed from are
not deposited. Where the legend and the code name different tests or quantities, both were run. For Figure 7, the claims file's choice of the unpaired tests as main readings (rank-sum for saline against CNO, unpaired t for Vgat against Vglut2) is asked of a reader.

What was found about the data. The dandiset's files store the z planes of each brain as
frames of an ImageSeries with a "rate" of 4.0, the z step in micrometres, and give no voxel size,
channel wavelengths or description; they are uncompressed and unchunked, 1.38 TB in all. The
consummatory (Mount) brains have session dates in February 2024, a year after the others, and
lower TRAP intensities in the middle plane. In the Figshare tables, `Time_in_Isolation_zone_per_10min`
holds the mean isolation visit in seconds, `Average_Isolation_zone_duration` of the Figure 7
table is in frames for three of the four groups, and the arrays named `norm_zscore` are rescaled
(for ejaculation, by each cell's SD over -5 to 0 s), so they do not give the table's magnitudes.
The deposited decay lengths follow a definition the Methods do not give (from the peak, with 120 s for a trace that never returns below 0), the clustering's chosen neighbours (30) and silhouette (0.474) differ from the Methods'
(55, 0.405), and the HCR populations count only cells with one of the two markers. The 14 brains match the Figure 2 subject table on Figshare by name and group, but the NWB files lack the orientation that table gives. For place preference, the legend's p-values for post-ejaculatory against appetitive hM3Dq (1, 1, 0.8472) are not those the test gives (0.162, 0.696, 0.0615).